# DM_CCP — Data Mining Project
### NED University | CT-377 | Fall 2025

In [3]:
# ============================================================
# STEP 1 — INSTALL & IMPORT ALL LIBRARIES
# ============================================================
!pip install requests beautifulsoup4 nltk textblob wordcloud scikit-learn transformers matplotlib pandas

import requests
from bs4 import BeautifulSoup
import pandas as pd
import nltk
import string
import time
import random
import matplotlib
matplotlib.use("Agg")          # headless backend — works on Colab AND Jupyter
import matplotlib.pyplot as plt
from textblob import TextBlob
from wordcloud import WordCloud
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from transformers import pipeline

nltk.download("stopwords", quiet=True)
nltk.download("punkt",     quiet=True)
from nltk.corpus import stopwords

print(" All libraries loaded.")

 All libraries loaded.


In [4]:
# ============================================================
# STEP 2 — USER INPUT
# ============================================================
keyword = input("Enter keyword (e.g., Machine Learning): ").strip()
print(f"Keyword set to: {keyword}")

Enter keyword (e.g., Machine Learning): machine learning
Keyword set to: machine learning


In [5]:
# ============================================================
# STEP 3 — DATA COLLECTION
# ============================================================

import requests, time, string, pandas as pd
from bs4 import BeautifulSoup
from nltk.corpus import stopwords

STOP = set(stopwords.words("english"))

def clean_text(text):
    text = str(text).lower().replace("\n", " ")
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [w for w in text.split() if w not in STOP]
    return " ".join(tokens)[:2000]

# ---------- SOURCE 1: Wikipedia ----------
def scrape_wikipedia(kw):
    results = []
    variants = [
    kw,
    kw + " algorithms",
    kw + " applications",
    kw + " techniques",
    kw + " examples"
]
    headers = {"User-Agent": "Mozilla/5.0 (educational-project)"}

    for term in variants:
        try:
            slug = term.strip().replace(" ", "_").title()
            url  = f"https://en.wikipedia.org/wiki/{slug}"
            r    = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(r.text, "html.parser")

            h1   = soup.find("h1")
            title = h1.text if h1 else term

            paras = soup.find_all("p")
            text  = " ".join(p.get_text() for p in paras if len(p.get_text()) > 50)

            if not text:
                continue

            results.append({
                "url": url,
                "title": title,
                "author": "Wikipedia Contributors",
                "date": "2024-01-01",
                "source": "Wikipedia",
                "raw_text": text[:3000],
                "comments": "Community-edited reference article. Neutral encyclopedic tone."
            })

            time.sleep(0.8)

        except Exception as e:
            print(f"  Wikipedia skip ({term}): {e}")

    return results


# ---------- SOURCE 2: Dev.to ----------
def scrape_devto(kw):
    results = []
    tag = kw.replace(" ", "").lower()

    try:
        r = requests.get(f"https://dev.to/api/articles?tag={tag}&per_page=30", timeout=12)

        for art in r.json()[:6]:
            try:
                detail = requests.get(f"https://dev.to/api/articles/{art['id']}", timeout=10).json()
                body   = detail.get("body_markdown") or art.get("description", "")

                results.append({
                    "url": art.get("url", ""),
                    "title": art.get("title", "No Title"),
                    "author": art.get("user", {}).get("name", "Unknown"),
                    "date": str(art.get("published_at", "Unknown"))[:10],
                    "source": "Dev.to",
                    "raw_text": body,
                    "comments": f"Reactions: {art.get('positive_reactions_count',0)}, Comments: {art.get('comments_count',0)}"
                })

                time.sleep(0.5)

            except Exception as e:
                print(f"  Dev.to article skip: {e}")

    except Exception as e:
        print(f"  Dev.to API error: {e}")

    return results


# ---------- SOURCE 3: HackerNews ----------
def scrape_hackernews(kw):
    results = []

    try:
        api = f"https://hn.algolia.com/api/v1/search?query={kw}&tags=story&hitsPerPage=35"
        r   = requests.get(api, timeout=12)

        for hit in r.json().get("hits", [])[:8]:
            story_url = hit.get("url", "")
            body = hit.get("story_text") or ""

            if story_url and not body:
                try:
                    sr = requests.get(story_url, headers={"User-Agent":"Mozilla/5.0"}, timeout=8)
                    ss = BeautifulSoup(sr.text, "html.parser")
                    body = " ".join(p.get_text() for p in ss.find_all("p"))[:3000]
                except:
                    body = hit.get("title", "")

            results.append({
                "url": story_url or f"https://news.ycombinator.com/item?id={hit.get('objectID','')}",
                "title": hit.get("title", "No Title"),
                "author": hit.get("author", "Unknown"),
                "date": str(hit.get("created_at", "Unknown"))[:10],
                "source": "HackerNews",
                "raw_text": body if body else hit.get("title", ""),
                "comments": f"Points: {hit.get('points',0)}, Comments: {hit.get('num_comments',0)}"
            })

            time.sleep(0.4)

    except Exception as e:
        print(f"  HackerNews API error: {e}")

    return results


# ---------- RUN ----------
print(f"\n🔍 Collecting blogs for: '{keyword}'\n")

print("[1/3] Wikipedia...")
wiki_data = scrape_wikipedia(keyword)
print(f"  {len(wiki_data)} articles")

print("[2/3] Dev.to...")
devto_data = scrape_devto(keyword)
print(f"  {len(devto_data)} articles")

print("[3/3] HackerNews...")
hn_data = scrape_hackernews(keyword)
print(f"  {len(hn_data)} articles")

all_data = wiki_data + devto_data + hn_data


# ---------- FALLBACK ----------
if len(all_data) < 10:
    print("Warning: Less than 10 articles collected. Consider increasing limits.")

    all_data = [
        {"url":"fb1","title":"Intro to Machine Learning","author":"John Doe","date":"2024-01-01","source":"Backup",
         "raw_text":"Machine learning is a subset of AI that teaches computers to learn from data without explicit programming.",
         "comments":"Very well written!"},
        {"url":"fb2","title":"Big Data Explained","author":"Jane Smith","date":"2024-02-15","source":"Backup",
         "raw_text":"Big data refers to datasets too large for traditional systems.",
         "comments":"Helpful explanation."}
    ]


df = pd.DataFrame(all_data)
df["clean_text"] = df["raw_text"].apply(clean_text)

print("\nData Collection Complete!")
print(f"Total articles : {len(df)}")
print(f"Sources        : {df['source'].value_counts().to_dict()}")
print(f"DataFrame shape: {df.shape}")

df[["title","author","date","source"]].head(15)


🔍 Collecting blogs for: 'machine learning'

[1/3] Wikipedia...
  1 articles
[2/3] Dev.to...
  6 articles
[3/3] HackerNews...
  8 articles

Data Collection Complete!
Total articles : 15
Sources        : {'HackerNews': 8, 'Dev.to': 6, 'Wikipedia': 1}
DataFrame shape: (15, 8)


,title,author,date,source
0,Machine learning,Wikipedia Contributors,2024-01-01,Wikipedia
1,My security scanner scored 0 out of 485. So I ...,ithiria894,2026-05-03,Dev.to
2,How to Build a Local Agentic Search Pipeline T...,Alan West,2026-05-03,Dev.to
3,Stop Reward Hacking Before It Breaks Your Mode...,Giovan Ruiz Vazquez,2026-05-03,Dev.to
4,What Happens When You Force All Math Through O...,Biswajyoti Nath,2026-05-03,Dev.to
5,When Generic Benchmarks Fail: Building a Sales...,Nati A,2026-05-02,Dev.to
6,"Open-source AI I'm watching: DeepSeek V4, Vibe...",盛永裕介,2026-05-02,Dev.to
7,Machine Learning Crash Course,matant,2018-03-01,HackerNews
8,Machine Learning 101 slidedeck: 2 years of hea...,flor1s,2017-12-14,HackerNews
9,The Chaostron: An Important Advance in Learnin...,abrax3141,2022-10-15,HackerNews


In [6]:
# ============================================================
# STEP 4 — NLP ANALYSIS
# Keyword extraction · Topic modeling · Tone · Motive
# ============================================================

# --- TF-IDF Keyword Extraction ---
vectorizer = TfidfVectorizer(stop_words="english", max_features=50)
X = vectorizer.fit_transform(df["clean_text"])
features = vectorizer.get_feature_names_out()

def top_keywords(row):
    scores = sorted(zip(features, row), key=lambda x: x[1], reverse=True)
    return [w for w, _ in scores[:5]]

df["keywords"] = [top_keywords(row) for row in X.toarray()]

# --- LDA Topic Modeling ---
n_topics = min(3, len(df))
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(X)
topics = [[features[i] for i in topic.argsort()[-5:]] for topic in lda.components_]
print(" Discovered Topics:")
for i, t in enumerate(topics):
    print(f"  Topic {i+1}: {t}")

# --- Motive Detection ---
def detect_motive(text):
    t = text.lower()
    if any(p in t for p in ["how to", "step by step", "tutorial", "guide"]):
        return "Tutorial"
    elif any(p in t for p in ["buy", "pricing", "discount", "offer", "sale"]):
        return "Promotional"
    elif any(p in t for p in ["research", "study", "findings", "paper", "journal"]):
        return "Academic"
    elif any(p in t for p in ["i think", "in my opinion", "i believe", "personally"]):
        return "Opinion"
    return "Informative"

# --- Tone Detection (Subjectivity via TextBlob) ---
def detect_tone(text):
    subjectivity = TextBlob(str(text)).sentiment.subjectivity
    return "Subjective/Opinion" if subjectivity > 0.5 else "Objective/Factual"

df["motive"] = df["raw_text"].apply(detect_motive)
df["tone"]   = df["raw_text"].apply(detect_tone)

print("\n Motive Distribution:", df["motive"].value_counts().to_dict())
print(" Tone Distribution:  ", df["tone"].value_counts().to_dict())
df[["title","source","motive","tone","keywords"]].head(12)

 Discovered Topics:
  Topic 1: ['model', 'rewardguard', 'loop', 'search', 'reward']
  Topic 2: ['like', 'years', 'structure', 'machine', 'learning']
  Topic 3: ['best', 'work', 'april', 'models', 'new']

 Motive Distribution: {'Academic': 5, 'Informative': 5, 'Tutorial': 4, 'Opinion': 1}
 Tone Distribution:   {'Objective/Factual': 11, 'Subjective/Opinion': 4}


,title,source,motive,tone,keywords
0,Machine learning,Wikipedia,Academic,Objective/Factual,"[learning, machine, algorithms, data, company]"
1,My security scanner scored 0 out of 485. So I ...,Dev.to,Academic,Objective/Factual,"[descriptions, tool, reads, text, use]"
2,How to Build a Local Agentic Search Pipeline T...,Dev.to,Tutorial,Objective/Factual,"[search, loop, model, running, context]"
3,Stop Reward Hacking Before It Breaks Your Mode...,Dev.to,Tutorial,Subjective/Opinion,"[reward, rewardguard, detection, training, agent]"
4,What Happens When You Force All Math Through O...,Dev.to,Academic,Objective/Factual,"[structure, built, like, think, 10]"
5,When Generic Benchmarks Fail: Building a Sales...,Dev.to,Tutorial,Objective/Factual,"[week, retail, agent, 10, benchmark]"
6,"Open-source AI I'm watching: DeepSeek V4, Vibe...",Dev.to,Opinion,Objective/Factual,"[april, model, window, context, week]"
7,Machine Learning Crash Course,HackerNews,Informative,Objective/Factual,"[best, building, models, real, model]"
8,Machine Learning 101 slidedeck: 2 years of hea...,HackerNews,Informative,Objective/Factual,"[years, machine, learning, 10, accuracy]"
9,The Chaostron: An Important Advance in Learnin...,HackerNews,Informative,Subjective/Opinion,"[learning, 10, accuracy, agent, ago]"


In [7]:
# ============================================================
# STEP 5 — LLM COMMENT GENERATION (GPT-2, no API key needed)
# ============================================================
print("Loading GPT-2 for comment generation (first run downloads ~500 MB)...")
generator = pipeline("text-generation", model="gpt2", pad_token_id=50256)

def generate_comment(title):
    if not isinstance(title, str) or title in ("Error", "No Title"):
        return "Interesting read!"
    prompt = f"Write a short user comment about this blog post: {title[:80]}"
    try:
        res = generator(prompt, max_new_tokens=25, do_sample=True,
                        temperature=0.7, top_p=0.9, return_full_text=False)
        return res[0]["generated_text"].replace("\n", " ").strip()
    except:
        return "Great article!"

df["generated_comment"] = df["title"].apply(generate_comment)
print(" Comments generated.")
df[["title","generated_comment"]].head(10)

Loading GPT-2 for comment generation (first run downloads ~500 MB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Passing `generation_config` together with generation-related arguments=({'top_p', 'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=25) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/m

 Comments generated.


,title,generated_comment
0,Machine learning,for Python. The goal is to learn how to use Ma...
1,My security scanner scored 0 out of 485. So I ...,"and found a bunch of data from GPT-2, and it's..."
2,How to Build a Local Agentic Search Pipeline T...,", by John B. Stein This post was originally p..."
3,Stop Reward Hacking Before It Breaks Your Mode...,RewardGuard is a free web-based reward managem...
4,What Happens When You Force All Math Through O...,(http://www.youtube.com/watch?v=3mHg7HdDtX8)
5,When Generic Benchmarks Fail: Building a Sales...,pping the Results to Get to Your Goals. The f...
6,"Open-source AI I'm watching: DeepSeek V4, Vibe...",", and I want to hear from you! I'm also looki..."
7,Machine Learning Crash Course,Machine learning crash course. Machine learni...
8,Machine Learning 101 slidedeck: 2 years of hea...,"work with machine learning. The problem is, if..."
9,The Chaostron: An Important Advance in Learnin...,"In a series of short posts, I will explore the..."


In [8]:
# ============================================================
# STEP 6 — SENTIMENT ANALYSIS ON COMMENTS
# + POPULARITY PROXY (comment count simulation)
# ============================================================

def get_sentiment(text):
    score = TextBlob(str(text)).sentiment.polarity
    if score >  0.1: return "Positive"
    if score < -0.1: return "Negative"
    return "Neutral"

# Analyse BOTH scraped comments AND generated comments
df["scraped_sentiment"]   = df["comments"].apply(get_sentiment)
df["generated_sentiment"] = df["generated_comment"].apply(get_sentiment)
df["sentiment_score"]     = df["generated_comment"].apply(
    lambda x: TextBlob(str(x)).sentiment.polarity)

# Popularity proxy: higher absolute sentiment → more engagement
random.seed(42)
df["comment_count"] = df["sentiment_score"].apply(
    lambda x: int(abs(x) * 100) + random.randint(5, 20))

print(" Generated Comment Sentiment:", df["generated_sentiment"].value_counts().to_dict())
print(" Scraped Comment Sentiment:  ", df["scraped_sentiment"].value_counts().to_dict())
df[["title","scraped_sentiment","generated_sentiment","sentiment_score","comment_count"]].head(12)

 Generated Comment Sentiment: {'Neutral': 8, 'Positive': 7}
 Scraped Comment Sentiment:   {'Neutral': 15}


,title,scraped_sentiment,generated_sentiment,sentiment_score,comment_count
0,Machine learning,Neutral,Neutral,0.000000,8
1,My security scanner scored 0 out of 485. So I ...,Neutral,Neutral,0.000000,5
2,How to Build a Local Agentic Search Pipeline T...,Neutral,Positive,0.375000,50
3,Stop Reward Hacking Before It Breaks Your Mode...,Neutral,Positive,0.400000,52
4,What Happens When You Force All Math Through O...,Neutral,Neutral,0.000000,12
5,When Generic Benchmarks Fail: Building a Sales...,Neutral,Positive,0.166667,25
6,"Open-source AI I'm watching: DeepSeek V4, Vibe...",Neutral,Positive,0.700000,78
7,Machine Learning Crash Course,Neutral,Neutral,0.000000,7
8,Machine Learning 101 slidedeck: 2 years of hea...,Neutral,Neutral,0.000000,18
9,The Chaostron: An Important Advance in Learnin...,Neutral,Positive,0.200000,26


In [9]:
# ============================================================
# STEP 7 — VISUALIZATIONS  (5 charts + 1 word cloud)
# ============================================================
import os
os.makedirs("charts", exist_ok=True)

# ── Chart 1: Sentiment Pie Chart ──────────────────────────
fig, ax = plt.subplots(figsize=(6,6))
counts = df["generated_sentiment"].value_counts()
colors = {"Positive":"#4CAF50","Neutral":"#FFC107","Negative":"#F44336"}
c_list = [colors.get(k,"#999") for k in counts.index]
ax.pie(counts, labels=counts.index, autopct="%1.1f%%", colors=c_list, startangle=140)
ax.set_title("Sentiment Distribution of Generated Comments", fontsize=13)
plt.tight_layout()
plt.savefig("charts/chart1_sentiment_pie.png", dpi=150)
plt.show()
print("Saved chart1_sentiment_pie.png")

# ── Chart 2: Sentiment vs Popularity Scatter ─────────────
fig, ax = plt.subplots(figsize=(7,5))
color_map = {"Positive":"green","Neutral":"orange","Negative":"red"}
for sent, grp in df.groupby("generated_sentiment"):
    ax.scatter(grp["sentiment_score"], grp["comment_count"],
               label=sent, color=color_map.get(sent,"gray"), s=80, alpha=0.8)
ax.set_xlabel("Sentiment Score")
ax.set_ylabel("Comment Count (Popularity Proxy)")
ax.set_title("Sentiment Score vs Popularity")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("charts/chart2_sentiment_vs_popularity.png", dpi=150)
plt.show()
print("Saved chart2_sentiment_vs_popularity.png")

# ── Chart 3: Motive Distribution Bar Chart ───────────────
fig, ax = plt.subplots(figsize=(7,4))
motive_counts = df["motive"].value_counts()
motive_counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
ax.set_title("Blog Motive Distribution")
ax.set_xlabel("Motive")
ax.set_ylabel("Number of Blogs")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("charts/chart3_motive_bar.png", dpi=150)
plt.show()
print("Saved chart3_motive_bar.png")

# ── Chart 4: Tone Distribution Bar Chart ─────────────────
fig, ax = plt.subplots(figsize=(6,4))
df["tone"].value_counts().plot(kind="bar", ax=ax, color=["#2196F3","#FF9800"], edgecolor="white")
ax.set_title("Tone Comparison Across Sources")
ax.set_xlabel("Tone")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("charts/chart4_tone_bar.png", dpi=150)
plt.show()
print("Saved chart4_tone_bar.png")

# ── Chart 5: Top Keywords Bar Chart ─────────────────────
from collections import Counter
all_kw = [kw for kws in df["keywords"] for kw in kws]
top_kw = Counter(all_kw).most_common(10)
kw_words, kw_freq = zip(*top_kw)
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(kw_words, kw_freq, color="mediumpurple")
ax.set_title("Top 10 Keywords Across All Blogs")
ax.set_xlabel("Frequency")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("charts/chart5_top_keywords.png", dpi=150)
plt.show()
print("Saved chart5_top_keywords.png")

# ── Word Cloud ────────────────────────────────────────────
all_text = " ".join(df["clean_text"].dropna())
wc = WordCloud(width=800, height=400, background_color="white",
               colormap="viridis", max_words=100).generate(all_text)
fig, ax = plt.subplots(figsize=(12,5))
ax.imshow(wc, interpolation="bilinear")
ax.axis("off")
ax.set_title("Word Cloud — Most Common Blog Keywords", fontsize=14)
plt.tight_layout()
plt.savefig("charts/wordcloud.png", dpi=150)
plt.show()
print("Saved wordcloud.png")

print("\n All 5 charts + word cloud saved in /charts/")

Saved chart1_sentiment_pie.png
Saved chart2_sentiment_vs_popularity.png
Saved chart3_motive_bar.png
Saved chart4_tone_bar.png
Saved chart5_top_keywords.png
Saved wordcloud.png

 All 5 charts + word cloud saved in /charts/


In [10]:
# ============================================================
# STEP 8 — FINAL SUMMARY TABLE
# ============================================================
print("\n===== PROJECT SUMMARY =====")
print(f"Keyword          : {keyword}")
print(f"Total Blogs      : {len(df)}")
print(f"Sources          : {df["source"].value_counts().to_dict()}")
print(f"Motives Found    : {df["motive"].value_counts().to_dict()}")
print(f"Tones Found      : {df["tone"].value_counts().to_dict()}")
print(f"Sentiment (gen.) : {df["generated_sentiment"].value_counts().to_dict()}")
print(f"Avg Popularity   : {df["comment_count"].mean():.1f} (simulated)")
print("\nSample Generated Comments:")
for _, row in df[["title","generated_comment"]].head(5).iterrows():
    print(f"  [{row["title"][:40]}] → {row["generated_comment"]}")
print("\n Pipeline complete!")


===== PROJECT SUMMARY =====
Keyword          : machine learning
Total Blogs      : 15
Sources          : {'HackerNews': 8, 'Dev.to': 6, 'Wikipedia': 1}
Motives Found    : {'Academic': 5, 'Informative': 5, 'Tutorial': 4, 'Opinion': 1}
Tones Found      : {'Objective/Factual': 11, 'Subjective/Opinion': 4}
Sentiment (gen.) : {'Neutral': 8, 'Positive': 7}
Avg Popularity   : 26.2 (simulated)

Sample Generated Comments:
  [Machine learning] → for Python. The goal is to learn how to use Machine Learning to improve your Python code. In this blog post, I
  [My security scanner scored 0 out of 485.] → and found a bunch of data from GPT-2, and it's just what I expected. So far I've found
  [How to Build a Local Agentic Search Pipe] → , by John B. Stein  This post was originally published on The Conversation. Read the original article.
  [Stop Reward Hacking Before It Breaks You] → RewardGuard is a free web-based reward management tool that lets you organize your rewards into categories, and then 

In [11]:
import nbformat
from google.colab import _message

# Get current notebook
nb_dict = _message.blocking_request('get_ipynb')['ipynb']
nb = nbformat.from_dict(nb_dict)

# 🔥 Remove widget metadata from notebook level
nb.metadata.pop("widgets", None)

# 🔥 Remove widget metadata from each cell
for cell in nb.cells:
    if "metadata" in cell:
        cell["metadata"].pop("widgets", None)

# 🔥 OPTIONAL: clear outputs (very important sometimes)
for cell in nb.cells:
    if "outputs" in cell:
        cell["outputs"] = []
    if "execution_count" in cell:
        cell["execution_count"] = None

# Save cleaned notebook
with open("Blog_analysis_clean.ipynb", "w", encoding="utf-8") as f:
    nbformat.write(nb, f)

print("Clean notebook saved!")

Clean notebook saved!


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/Blog_analysis.ipynb" .

In [12]:
!jupyter nbconvert --to notebook --ClearOutputPreprocessor.enabled=True Blog_analysis.ipynb

[NbConvertApp] WARNING | pattern 'Blog_analysis.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
 